<a target="_parent" href="https://colab.research.google.com/github/gretelai/gretel-blueprints/blob/main/docs/notebooks/data-designer/document-intelligence/image-summarization.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# 🎨 Data Designer: Image Summarization

In this notebook, we demonstrate how to create detailed summaries from document images using **Data Designer**. This workflow processes document images with vision-language models to generate comprehensive, structured summaries that capture key information from visual content.

## What This Notebook Does

This notebook shows how to:
- **Process document images** from existing datasets using vision-language models
- **Generate comprehensive summaries** from visual document content
- **Extract structured information** from images including text, tables, and figures
- **Create markdown-formatted summaries** that organize document content logically
- **Scale image summarization** across large document collections

The workflow transforms document images into structured text summaries suitable for search indexing, content analysis, and document processing pipelines.

<br>

### 💾 Install `gretel-client` and dependencies

In [ ]:
%%capture
%pip install -U gretel_client datasets rich

## ⚙️ Setup and Configuration

Let's start by setting up the required imports and initializing our Data Designer client with a custom model configuration. We'll use NVIDIA's API to power our vision-language model for processing document images.

In [ ]:
import io
import os
import json
import base64
import random
import uuid

import pandas as pd
from pydantic import BaseModel, Field
from datasets import load_dataset
from itertools import product

import rich
from rich.panel import Panel
from rich.markdown import Markdown

from gretel_client.navigator_client import Gretel
from gretel_client.data_designer.params import ModelConfig, GenerationParameters
import gretel_client.data_designer.params as P
import gretel_client.data_designer.columns as C

# The Gretel object is the SDK's main entry point for interacting with Gretel's API.
gretel = Gretel(api_key="prompt")

# Create an API key connection to NVIDIA's API for vision-language model access
nvbuild_connection_id = gretel.data_designer.create_api_key_connection(
    name="nvbuild-connection",
    api_base="https://integrate.api.nvidia.com/v1/",
    api_key=os.getenv("NVBUILD_API_KEY", "")
)

## 📄 Document Image Processing

We'll use an existing document-image dataset as the foundation for our summarization workflow. The dataset contains various document types including PDFs, research papers, and forms that we'll process to extract structured summaries.

In a real-world scenario, you might:
- Convert PDF pages to images using tools like `pdf2image`
- Process scanned documents or forms
- Work with screenshots of web pages or applications
- Use existing document image datasets for analysis

Below, we'll prepare the images for processing by resizing them and converting them to a format suitable for vision-language models.

> **⚠️ Important Note**: The image dataset used in this notebook is formatted specifically for the Mistral Medium vision-language model. If you change the vision model (e.g., to GPT-4V, Claude Vision, or other VLMs), you'll need to format the dataset accordingly, as different models may have different image input requirements, size limitations, or encoding formats.

In [ ]:
# Configuration for image processing
img_count = 512  # Number of images to process
chat_image_height = 512  # Standardized height for model input

# Dataset configuration - using the vidore/colpali_train_set dataset
img_dataset_cfg = {
    "path": "vidore/colpali_train_set",
    "split": "train",
    "streaming": True
}

def resize(image, height: int):
    """Resize image while maintaining aspect ratio."""
    original_width, original_height = image.size
    width = int(original_width * (height / original_height))
    return image.resize((width, height))

def image_to_chat(record, height: int) -> str:
    """Convert PIL image to base64 format for chat template usage."""
    image = resize(record["image"], height)
    img_buffer = io.BytesIO()
    
    image.save(img_buffer, format="PNG")
    byte_data = img_buffer.getvalue()
    base64_encoded_data = base64.b64encode(byte_data)
    base64_string = base64_encoded_data.decode("utf-8")

    return record | {
        "chat_image": f'<img src="data:image/png;base64,{base64_string}" />',
        "uuid": str(uuid.uuid4())
    }

# Load and process the dataset
print("Loading and processing document images...")
img_dataset_iter = iter(load_dataset(**img_dataset_cfg).map(image_to_chat, fn_kwargs={"height": chat_image_height}))
img_dataset = pd.DataFrame([next(img_dataset_iter) for _ in range(img_count)])

print(f"Loaded {len(img_dataset)} images with columns: {list(img_dataset.columns)}")

## 🧠 Initialize Data Designer with Vision-Language Model

Now we'll create a Data Designer instance configured with a vision-language model. We're using **Mistral Medium** through NVIDIA's API, which provides strong multimodal capabilities for processing both text and images.

The model configuration includes:
- **Temperature**: 0.7 for balanced creativity and consistency
- **Top-p**: 0.98 for diverse but coherent outputs
- **Max tokens**: 16,000 for detailed responses

In [ ]:
# Initialize Data Designer with custom model configuration
query_designer = gretel.data_designer.new(
    model_suite="bring-your-own", 
    model_configs=[
        ModelConfig(
            alias="vlm",  # Vision-Language Model alias
            model_name="mistralai/mistral-medium-3-instruct",
            generation_parameters=GenerationParameters(
                temperature=0.7, 
                top_p=0.98, 
                max_tokens=16_000
            ),
            connection_id=nvbuild_connection_id
        ),
    ]
)

## 📝 Generate Document Summaries

This is the core functionality of our pipeline: creating detailed summaries of document images using vision-language models. The summaries extract and organize key information from visual content into structured text format.

The vision-language model will:
- **Analyze document content** systematically from top to bottom
- **Extract key information** including text, tables, figures, and document structure
- **Generate markdown-formatted summaries** that capture the document's essence
- **Organize content logically** for easy processing and understanding

These summaries serve as structured representations of document content that can be used for search indexing, content analysis, and document processing workflows.

In [ ]:
# Add the seed dataset containing our processed images
query_designer.with_seed_dataset(
    pd.DataFrame(img_dataset)[["uuid", "image_filename", "chat_image", "page", "options", "source"]],
    sampling_strategy="ordered",
    with_replacement=True
)

# Add a column to generate detailed document summaries
query_designer.add_column(
    name="summary",
    type="llm-code",
    prompt="""\
{{chat_image}}

Provide a detailed summary of the content in this image in Markdown format. Start from the top of the image and then describe it from top to bottom.
Place a summary at the bottom.
""",
    model_alias="vlm",
    output_format="markdown"
)

In [ ]:
# Generate a preview to test our document summarization
preview = query_designer.preview(verbose_logging=True)

## 👀 Preview Document Summaries

Let's examine the quality of our generated summaries by comparing them with the original document images. This step validates that our vision-language model is effectively extracting and structuring information from the visual content.

The summaries should provide:
- **Comprehensive coverage** of the document's main content
- **Structured markdown format** for easy processing
- **Accurate information extraction** from text, tables, and figures
- **Logical organization** that reflects the document's structure

In [ ]:
# View sample outputs - compare original image with generated summary
index = 7
comparison_dataset = preview.dataset.df.merge(
    pd.DataFrame(img_dataset)[["uuid", "image"]], 
    how="left", 
    on="uuid"
)

print("Original document image:")
display(resize(comparison_dataset.image[index], chat_image_height))

print("\nGenerated summary:")
rich.print(Panel(comparison_dataset.summary[index], title="Generated Summary"))

## 🚀 Generate Full Summarization Dataset

Once you're satisfied with the preview results, scale up to generate summaries for your complete document collection. This processes all configured document images through our summarization pipeline.

The full workflow will:
- **Process all document images** with vision-language models
- **Generate comprehensive summaries** for each document
- **Extract structured information** from visual content
- **Output organized summaries** ready for downstream applications

**Note**: Processing time depends on the number of images and model response times. Monitor the workflow progress through the Gretel console.

In [ ]:
# Generate diverse user personas
query_designer.add_column(
    name="query_persona",
    type="person"
)

# Create realistic scenarios explaining the persona's interest in the document
query_designer.add_column(
    name="query_scenario", 
    type="llm-text",
    prompt="""\
<persona>
{{query_persona}}
</persona>

<document>
{{ summary }}
</document>

Provide a *single*, short, three-sentence scenario which explains why the <persona> is interested in the information contained in the <document> given above.
Critically, {{query_persona.first_name}} has never seen the <document> before, but they are interested in the kind of information it contains. 
""",
    model_alias="vlm"
)

# Generate natural language queries based on the scenario
query_designer.add_column(
    name="query",
    type="llm-text",
    prompt="""\
<document>
{{ summary }}
</document>

{{ query_scenario }}

Because of this, {{ query_persona.first_name }} is searching for information like that contained in the <document> from an AI database's natural language search tool. 
The AI database is able to understand {{ query_persona.first_name }}'s query perfectly and will retrieve the <document> and answer their question or 
query. {{ query_persona.first_name}} types a query into the AI database's search tool's text box. What does {{ query_persona.first_name}} type? Only respond with the
text written in the text box.
""",
    model_alias="vlm"
)

## 📊 Process and Export Summaries

After the summarization workflow completes, we can retrieve the generated summaries and export them for use in downstream applications. This step focuses on organizing and formatting the extracted document summaries.

The processing includes:
- **Retrieving generated summaries** from the completed workflow
- **Organizing summary data** with document metadata
- **Formatting for export** in standard formats
- **Preparing for integration** with existing document processing pipelines

In [ ]:
# Define the evaluation structure
n_judges = 3

class DocumentRelevance(BaseModel):
    is_relevant: bool = Field(..., description="True if the document is relevant to the query, False otherwise.")

# Create multiple judges for robust evaluation
for judge_idx in range(n_judges):
    # Judge based on text summary
    query_designer.add_column(
        name=f"judge_text_query_relevance_{judge_idx}",
        type="llm-structured",
        prompt="""\
<document>
{{ summary }}
</document>
<query>
{{ query }}
</query>

# Is the information in <document> relevant to the <query>? 
""",
        output_format=DocumentRelevance,
        model_alias="vlm"
    )

    # Judge based on original image
    query_designer.add_column(
        name=f"judge_image_query_relevance_{judge_idx}",
        type="llm-structured",
        prompt="""\
{{ chat_image }}

<query>
{{ query }}
</query>

Is the information in scanned document image relevant to the <query>? 
""",
        output_format=DocumentRelevance,
        model_alias="vlm"
    )

# Build a consensus column requiring unanimous agreement
truth_statement = " and ".join(
    f"judge_{m}_query_relevance_{i}.is_relevant"
    for m, i in product(["text", "image"], range(n_judges))
) 

consensus_expression = f"""\
{{%- if {truth_statement} -%}}
True
{{%- else -%}}
False
{{%- endif -%}}
"""

query_designer.add_column(
    name="query_relevance_consensus",
    type="expression",
    expr=consensus_expression
)

In [ ]:
# Preview the complete pipeline with all components
preview = query_designer.preview()

In [ ]:
# Examine the preview results
preview.dataset.df

## 🚀 Generate Full Summarization Dataset

Once you're satisfied with the preview results, scale up to generate summaries for your complete document collection. This processes all configured document images through our summarization pipeline.

The full workflow will:
- **Process all document images** with vision-language models
- **Generate comprehensive summaries** for each document
- **Extract structured information** from visual content
- **Output organized summaries** ready for downstream applications

**Note**: Processing time depends on the number of images and model response times. Monitor the workflow progress through the Gretel console.

In [ ]:
# Submit the workflow for full dataset generation
workflow_run = query_designer.create(
    num_records=img_count,
    wait_until_done=False  # Set to True if you want to wait for completion
)

## 📊 Process and Export Summaries

After the summarization workflow completes, we can retrieve the generated summaries and export them for use in downstream applications. This step focuses on organizing and formatting the extracted document summaries.

The processing includes:
- **Retrieving generated summaries** from the completed workflow
- **Organizing summary data** with document metadata
- **Formatting for export** in standard formats
- **Preparing for integration** with existing document processing pipelines

In [ ]:
# Retrieve the completed dataset (update with your actual workflow run ID)
workflow_run_id = "wr_2yz2qSyebQwBDcdMDPtIbnTgNcp"

generated_dataset = gretel.workflows.get_workflow_run(workflow_run_id).dataset.df

In [ ]:
import re 
from datasets import Dataset, Image, Features, Value
from PIL.PngImagePlugin import PngImageFile
from PIL.JpegImagePlugin import JpegImageFile

# Define columns to retain in the final dataset
retained_columns = [
    "image",
    "query",
    "summary",
    "image_filename",
    "page",
    "options",
    "source",
    "query_persona",
]

dataset_filename = "query_generation_example.parquet"

# Helper function to clean query formatting
_quote_strip_re = re.compile(r"^(['\"])(.*)\1$")
def strip_outer_quotes(text: str) -> str:
    """Remove outer quotes from generated queries."""
    return _quote_strip_re.sub(r"\2", text)
    
# Process and filter the dataset
print("Processing and filtering dataset...")
synthetic_query_dataset = (
    generated_dataset
    .query("query_relevance_consensus == 'True'")  # Only keep high-quality Q&A pairs
    .merge(pd.DataFrame(img_dataset)[["uuid", "image"]], how="left", on="uuid")
    .assign(
        query=lambda df: df["query"].apply(strip_outer_quotes),
    )
    [retained_columns]
)

print(f"Generated {len(synthetic_query_dataset)} high-quality Q&A pairs")

# Save the dataset in Parquet format with proper image handling
(
    Dataset
    .from_dict(synthetic_query_dataset.to_dict(orient="list"), split="train")
    .cast_column("image", Image())
    .to_parquet(dataset_filename)
)

print(f"Dataset saved to {dataset_filename}")

## 🎯 Explore the Generated Summaries

Let's examine examples from our generated summaries to evaluate the quality of our document image summarization pipeline. This showcases how vision-language models can effectively:

- **Extract structured information** from visual document content
- **Generate coherent summaries** that capture key document elements
- **Organize content logically** in markdown format
- **Maintain accuracy** in information extraction

In [ ]:
# Load the saved dataset
dset = load_dataset("parquet", data_files=dataset_filename, split="train")
print(f"Loaded dataset with {len(dset)} Q&A pairs")

In [ ]:
# Display a sample Q&A pair with the original document
index = 300

print("🔍 Sample Q&A Pair:")
rich.print(Panel(dset["query"][index], title="💬 User Query"))
rich.print(Panel(Markdown(dset["summary"][index]), title="📝 Document Summary"))

print("\n📄 Original Document:")
display(dset["image"][index])

## 🎉 Next Steps

Congratulations! You've successfully created an image summarization pipeline using Data Designer. Your workflow generates detailed, structured summaries from document images. Here are some ideas for next steps:

### 🔄 **Iterate and Improve**
- Refine summarization prompts to better capture document structure and content
- Experiment with different vision-language models for improved accuracy
- Adjust summary formats to match your specific domain requirements

### 🎯 **Customize for Your Use Case**
- Replace the sample dataset with your own document images (PDFs, forms, reports)
- Modify summary templates for specific document types
- Add custom metadata fields that are important for your application

### 🚀 **Scale and Deploy**
- Process large document collections with batch workflows
- Integrate with existing document processing pipelines
- Use generated summaries for search indexing and retrieval systems
- Create content analysis and classification workflows

### 📊 **Advanced Summarization Applications**
- Generate structured data extractions from forms and invoices
- Create multi-level summaries (executive, technical, detailed)
- Build document classification systems based on summarized content
- Develop automated content tagging and categorization

### 💡 **Integration Opportunities**
- Combine with OCR pipelines for enhanced text extraction
- Create search indexes from generated summaries
- Build document similarity and comparison systems
- Develop automated content validation workflows

---

For more Data Designer examples and document intelligence use cases, check out the [Gretel Blueprints repository](https://github.com/gretelai/gretel-blueprints/tree/main/docs/notebooks/data-designer).